In [0]:
%pip install openpyxl

In [0]:
import pandas as pd
import openpyxl
import requests
import re #regex

In [0]:
# Path to download the API
path_munic = "https://ftp.ibge.gov.br/Perfil_Municipios/2024/Base_de_Dados/Base_MUNIC_2024_20251107.xlsx"
munic_file = "Base_MUNIC_2024.xlsx"

In [0]:
# Do the request to the IBGE APIs
file = requests.get(path_munic)

In [0]:
# Returns a exception if occur a error on the request
file.raise_for_status()

In [0]:
with open(munic_file, "wb") as f:
    f.write(file.content)

Workbook (wb) = the complete file

Worksheet (ws) = the individual tabbed pages inside the workbook

In [0]:
# Inspect available sheets
wb = openpyxl.load_workbook(munic_file, read_only=True)
print("Available sheets:", wb.sheetnames)

In [0]:
type(munic_file)

In [0]:
# Take the specific worksheet that we want
info_ws = wb["Informática e comunicação"]

In [0]:
for row in info_ws.iter_rows(max_row=5, values_only=True):
  print(row)

## Convert the Worksheets to Dataframe

- "Informatica e Comunicação", Governance and Dicionary worksheets

In [0]:
df_dict = pd.read_excel("Base_MUNIC_2024.xlsx", sheet_name="Dicionário", header=None)

In [0]:
df_info_ws = pd.read_excel(
    "Base_MUNIC_2024.xlsx",
    sheet_name="Informática e comunicação"
)

df_governance_ws = pd.read_excel(
    "Base_MUNIC_2024.xlsx",
    sheet_name="Governanca"
)

In [0]:
df_data = pd.merge(df_info_ws, df_governance_ws, on="Cod Munic", how="inner")

In [0]:
df_dict.head(50)

## Extract the Meaning of the Code Variables on the Sheets

- It´s keeped on the df_metabase_info_ws on the final of this section

In [0]:
def obtain_filled_cells(row):
    # Basically catch the values in a row from the dictionary sheet, to retire the columns vacances
    filled_cells = [str(val).strip() for val in row if pd.notna(val) and str(val).strip() != '']

    return filled_cells


def extract_var_code_meaning(row, var_pattern_list):
  """
    Extracts variable codes and their corresponding survey descriptions
    from a raw IBGE MUNIC dictionary row.

    Parameters:
        row (pd.Series): A single row from the unformatted dictionary sheet.
        var_pattern (str): The thematic variable prefix to match (e.g., 'MTIC', 'MGOV', 'MREH').

    Returns:
        dict: {'cod_variable': <str>, 'meaning': <str>} if a code matching the pattern is found,
              otherwise None.
    """

  # It´s used to save the pattern matched on the row using regex
  code_matches = list()

  filled_cells = obtain_filled_cells(row)

  # Verify if the row has values or not
  if not filled_cells:
      return None

  # Create the Regex Expressions
  regex_expr_list = [rf'^{pattern}\d+' for pattern in var_pattern_list]


  # Match if any cell have some pattern codes and add it in code_matches list
  for re_expr in regex_expr_list:
      for val in filled_cells:
        if re.match(re_expr, val, flags=re.IGNORECASE):
           code_matches.append(val)


  # If has any value with some of the code patterns
  if code_matches:
      cod_var = code_matches[0]

      # Find the description or what means of the MTIC code
      desc_cells = [val for val in filled_cells if val != cod_var]

      # If has one string value, get it. If has more, join the them. If doesn´t have, return None meaning to that code
      if desc_cells:
        if len(desc_cells) == 1:
          code_meaning = desc_cells[0]
        else:
          code_meaning = " - ".join(desc_cells) if desc_cells else None

      return {"cod_variable": cod_var, "meaning": code_meaning}

  # Else, returns None
  return None

In [0]:
# Get the meaning of the codes in the "Dicionario" worksheet (Ws)

code_patterns = ['MTIC', 'MGOV']
ws_meta_vars = [res for res in df_dict.apply(extract_var_code_meaning, axis=1, args=(code_patterns,)) if res is not None]

In [0]:
ws_meta_vars

In [0]:
# Transform on Dataframe
df_meta_vars = pd.DataFrame(ws_meta_vars)

In [0]:
df_meta_vars.info()

In [0]:
# Convert the value from the columns to String and keep them clean
df_meta_vars['cod_variable'] = df_meta_vars['cod_variable'].astype(str).str.strip().str.upper()
df_meta_vars['meaning'] = df_meta_vars['meaning'].astype(str).str.strip()

In [0]:
df_meta_vars.iloc[0]

In [0]:
df_meta_vars.iloc[357]

# Pre-Processing the data


In [0]:
df_data.head()

In [0]:
df_info_ws.info()

In [0]:
x = 0
for column in df_data.columns:
    print(column, end="\t\t")
    x += 1
    if (x % 7 == 0):
        print("\n")